# Day 26 — **Reasoning Trace: Dataclass Chain**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~30 min

Today's LangGraph lesson leaned on a *reasoning trace* — Thought → Action → Observation. That structure deserves a real type. A `@dataclass` gives you a typed, self-documenting record with `__init__`, `__repr__`, and `==` for free — perfect for the audit log a banking agent must produce.

Today:
1. `@dataclass ReasoningStep(thought, action, observation, timestamp)` with a `default_factory` timestamp.
2. A `ReasoningTrace` that chains steps and prints them.
3. `frozen=True` for tamper-proof audit records + JSON serialisation with `asdict`.
4. `__post_init__` validation so a malformed step can't exist.

## 1. The `ReasoningStep` dataclass

`@dataclass` writes the boilerplate. Use `field(default_factory=...)` for the timestamp — a plain default would be evaluated **once at class definition**, freezing every step to the same time. `default_factory` runs per instance.

In [1]:
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone

@dataclass
class ReasoningStep:
    thought: str
    action: str
    observation: str
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

s = ReasoningStep(
    thought="need Priya's overdraft limit",
    action="lookup_overdraft('priya')",
    observation="2000",
)
print(s)
print("timestamp set per-instance:", s.timestamp[:19])

ReasoningStep(thought="need Priya's overdraft limit", action="lookup_overdraft('priya')", observation='2000', timestamp='2026-07-14T04:16:39.522968+00:00')
timestamp set per-instance: 2026-07-14T04:16:39


☝ Four typed fields and a free `__repr__`. The `default_factory` lambda runs at *construction*, so each step stamps its own time — the classic mutable/`datetime`-default trap avoided. `thought`, `action`, `observation` document the ReAct triple in the type itself.

## 2. Chain steps into a trace

A trace is an ordered list of steps. Wrap it so you can `add()` and render it — the render is what goes in the log line an SRE reads at 3am.

In [2]:
@dataclass
class ReasoningTrace:
    goal: str
    steps: list = field(default_factory=list)

    def add(self, thought, action, observation):
        self.steps.append(ReasoningStep(thought, action, observation))
        return self                                   # chainable

    def render(self):
        lines = [f"GOAL: {self.goal}"]
        for i, s in enumerate(self.steps, 1):
            lines.append(f"  {i}. T:{s.thought} | A:{s.action} | O:{s.observation}")
        return "\n".join(lines)

t = ReasoningTrace(goal="sum overdrafts for Sarah and Priya")
t.add("get Sarah", "lookup('sarah')", "3000").add("get Priya", "lookup('priya')", "2000")
t.add("sum them", "calculator('3000+2000')", "5000")
print(t.render())

GOAL: sum overdrafts for Sarah and Priya
  1. T:get Sarah | A:lookup('sarah') | O:3000
  2. T:get Priya | A:lookup('priya') | O:2000
  3. T:sum them | A:calculator('3000+2000') | O:5000


☝ `steps` uses `default_factory=list` — again dodging the shared-mutable-default bug (a bare `[]` default is shared across *all* instances). `add()` returns `self`, so calls chain fluently, mirroring how the agent appends one reasoning step per loop iteration.

## 3. `frozen=True` + JSON — a tamper-proof audit record

For compliance, a logged step must be **immutable**. `frozen=True` makes assignment raise `FrozenInstanceError`. `asdict` then serialises the whole trace to plain dicts → JSON for storage.

In [3]:
import json
from dataclasses import FrozenInstanceError

@dataclass(frozen=True)
class AuditStep:
    thought: str
    action: str
    observation: str

a = AuditStep("check limit", "lookup", "2000")
try:
    a.observation = "9999"                            # attempt to falsify the record
except FrozenInstanceError as e:
    print("blocked mutation:", e)

print(json.dumps(asdict(a), indent=2))

blocked mutation: cannot assign to field 'observation'
{
  "thought": "check limit",
  "action": "lookup",
  "observation": "2000"
}


☝ `frozen=True` turns the record read-only — you literally cannot rewrite `observation` after the fact, which is what makes it trustworthy as evidence. `asdict` flattens the dataclass (recursively, including nested ones) into JSON-ready dicts, so the whole trace persists to a log store in one line.

## 4. `__post_init__` — validate on construction

`__post_init__` runs right after the generated `__init__`. Use it to reject a step that violates an invariant, so a malformed reasoning record can never enter the log in the first place.

In [4]:
@dataclass
class ValidatedStep:
    thought: str
    action: str
    observation: str

    def __post_init__(self):
        if not self.thought.strip():
            raise ValueError("thought must not be empty")
        if "(" not in self.action:
            raise ValueError(f"action must look like a call, got {self.action!r}")

print(ValidatedStep("need balance", "get_balance('AC-1')", "1200"))   # ok
try:
    ValidatedStep("", "noop", "x")
except ValueError as e:
    print("rejected:", e)

ValidatedStep(thought='need balance', action="get_balance('AC-1')", observation='1200')
rejected: thought must not be empty


☝ Validation lives in one place and fires at construction — impossible to build an invalid `ValidatedStep`. That's the dataclass superpower for agent plumbing: the *shape* and the *rules* of a reasoning step travel together with the data, so every producer in the codebase gets the same guarantees for free.

## 5. Recap — dataclasses model an agent's memory

| Feature | Why it matters for agent traces |
|---|---|
| `@dataclass` | free `__init__`/`__repr__`/`__eq__` for step records |
| `field(default_factory=...)` | per-instance timestamps & lists (no shared-mutable bug) |
| `frozen=True` | immutable, tamper-proof audit steps |
| `asdict` | one-line serialisation to JSON for log storage |
| `__post_init__` | reject malformed steps at construction |

A reasoning trace is an agent's short-term memory *and* its audit log. Modelling each step as a typed, validated, immutable record is what turns "the agent said so" into "here is exactly what the agent did, and it can't be edited.".